# 💱 Quantum Monte Carlo for Derivative Pricing
### Quantum for Finance — Quantum for Humanity

This notebook demonstrates **Quantum Amplitude Estimation (QAE)** for pricing European call options — providing a **quadratic speedup** over classical Monte Carlo methods.

**Learning source:** [IBM Quantum Learning](https://learning.quantum.ibm.com)

---

## Background

Classical Monte Carlo option pricing requires $O(1/\epsilon^2)$ samples to achieve $\epsilon$ precision.
Quantum Amplitude Estimation achieves $O(1/\epsilon)$ — a **quadratic speedup**.

For a European call option:
$$V = e^{-rT} \mathbb{E}[\max(S_T - K, 0)]$$

where $S_T$ is the stock price at maturity, $K$ is the strike price, $r$ is risk-free rate, $T$ is time to maturity.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit_finance.circuit.library import LogNormalDistribution
from qiskit_algorithms import IterativeAmplitudeEstimation, EstimationProblem
from qiskit.circuit.library import LinearAmplitudeFunction
from qiskit.primitives import Sampler

print('✅ Imports successful')

## Step 1: Define Option Parameters

In [ ]:
# Option parameters
S0 = 100.0    # Initial stock price
K  = 105.0    # Strike price
r  = 0.05     # Risk-free rate (annual)
sigma = 0.20  # Volatility
T  = 1.0      # Time to maturity (years)

# Classical Black-Scholes reference price
from scipy.stats import norm
d1 = (np.log(S0/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
d2 = d1 - sigma*np.sqrt(T)
bs_price = S0*norm.cdf(d1) - K*np.exp(-r*T)*norm.cdf(d2)
print(f'Black-Scholes Reference Price: ${bs_price:.4f}')

# Quantum circuit parameters
num_qubits = 3  # discretisation of stock price distribution

## Step 2: Load Stock Price Distribution into Quantum Circuit

In [ ]:
# Log-normal distribution parameters (risk-neutral measure)
mu_ln = (r - 0.5*sigma**2)*T
sigma_ln = sigma*np.sqrt(T)

# Stock price range for discretisation
S_min = S0 * np.exp(mu_ln - 3*sigma_ln)
S_max = S0 * np.exp(mu_ln + 3*sigma_ln)

# Build log-normal distribution circuit
distribution = LogNormalDistribution(
    num_qubits=num_qubits,
    mu=np.log(S0) + mu_ln,
    sigma=sigma_ln,
    bounds=(S_min, S_max)
)

# Plot the discretised distribution
x = np.linspace(S_min, S_max, 2**num_qubits)
plt.figure(figsize=(8, 4))
plt.bar(x, distribution.probabilities, width=(S_max-S_min)/2**num_qubits, 
        color='#8B5CF6', alpha=0.7, label='Quantum discretisation')
plt.axvline(K, color='#FF6B9D', linestyle='--', linewidth=2, label=f'Strike K=${K}')
plt.xlabel('Stock Price $S_T$'); plt.ylabel('Probability')
plt.title('Log-Normal Stock Price Distribution (Quantum Circuit)'); plt.legend()
plt.tight_layout(); plt.show()

## Step 3: Encode Payoff Function & Run Quantum Amplitude Estimation

In [ ]:
# Normalise payoff: max(S - K, 0) → [0, 1] for amplitude encoding
breakpoints = [S_min, K]
slopes      = [0.0, 1.0 / (S_max - K)]
offsets     = [0.0, 0.0]
f_min, f_max = 0.0, 1.0

# Linear amplitude function (encodes normalised call payoff)
european_call_payoff = LinearAmplitudeFunction(
    num_state_qubits=num_qubits,
    slope=slopes, offset=offsets,
    domain=(S_min, S_max),
    image=(f_min, f_max),
    breakpoints=breakpoints
)

# Build estimation problem
problem = EstimationProblem(
    state_preparation=distribution.compose(european_call_payoff, inplace=False),
    objective_qubits=[num_qubits]
)

# Run Iterative QAE
iae = IterativeAmplitudeEstimation(
    epsilon_target=0.01,
    alpha=0.05,
    sampler=Sampler()
)
result = iae.estimate(problem)

# Rescale to actual option price
conf_int = result.confidence_interval_processed
option_price = result.estimation_processed * (S_max - K) * np.exp(-r * T)

print(f'\n⚛️  Quantum (QAE) Option Price:   ${option_price:.4f}')
print(f'📊  Classical (Black-Scholes):     ${bs_price:.4f}')
print(f'📏  Confidence Interval:           [{conf_int[0]*(S_max-K)*np.exp(-r*T):.4f}, {conf_int[1]*(S_max-K)*np.exp(-r*T):.4f}]')

## 🌍 Humanitarian Application

Quantum Monte Carlo pricing can be applied to:
- **Climate insurance products** for smallholder farmers
- **Weather derivatives** protecting against crop failure
- **Parametric insurance** for disaster-prone communities

Faster, cheaper risk pricing → affordable insurance → financial resilience for underserved populations.

*Part of [Quantum for Humanity](https://github.com/vivekiniitm-stack/Quantum-for-humanity)*